In [0]:
# first download the excel reader 
%pip install openpyxl==3.1.5


Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
from pathlib import Path
from datetime import datetime, timezone
from openpyxl import load_workbook
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType


# Location of the uploaded workbook.
CATALOG = "sales_quality"
SCHEMA = "bronze_layer"
RAW_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data"
FILE_PATH = f"{RAW_PATH}/sales_practice.xlsx"
SHEET_NAME = "Sales"

# Full name of the Bronze table we will create.
TABLE_NAME = f"`{CATALOG}`.`{SCHEMA}`.`sales`"

# Display files currently stored in the volume.
display(dbutils.fs.ls(RAW_PATH))

path,name,size,modificationTime
dbfs:/Volumes/sales_quality/bronze_layer/raw_data/sales_practice.xlsx,sales_practice.xlsx,91347,1793373266000
dbfs:/Volumes/sales_quality/bronze_layer/raw_data/sales_reference.xlsx,sales_reference.xlsx,7770,1793373267000


# Professional Ingestion Process

The notebook follows a controlled sequence to load the Sales workbook into the Bronze layer.

1. **Set the configuration**
   - Define the catalog, schema, volume path, Excel file path, worksheet name and target table name.

2. **List the volume files**
   - Confirm that Databricks can see the uploaded workbook in the expected volume location.

3. **Check the source file**
   - Confirm that `sales_practice.xlsx` exists.
   - Stop if the file is missing.

4. **Check the worksheet**
   - Confirm that the required `Sales` worksheet exists inside the workbook.
   - Stop if the worksheet is missing.

5. **Create the destination schema**
   - Create `sales_quality.bronze_layer` if it does not already exist.

6. **Check the destination table**
   - Check whether `sales_quality.bronze_layer.sales` already exists.
   - Stop if it exists to prevent accidental overwriting or duplication.

7. **Read the Excel worksheet**
   - Read the headers and data rows.
   - Preserve missing values, invalid values and duplicates.

8. **Apply an explicit schema**
   - Define the Spark columns manually.
   - Keep Bronze business values as text so values such as `"five"` are preserved.

9. **Add ingestion metadata**
   - Add the source file, worksheet name, original Excel row number and ingestion timestamp.

10. **Write the Bronze Delta table**
    - Save the DataFrame as `sales_quality.bronze_layer.sales`.

11. **Reconcile the result**
    - Compare the number of source rows with the number of saved table rows.

12. **Preview the result**
    - Display the saved table and schema to confirm that ingestion worked correctly.


In [0]:
# Confirm that the source Excel file is available.
if not Path(FILE_PATH).is_file():
    raise FileNotFoundError(f"Workbook not found: {FILE_PATH}")

# The catalog and raw-data volume should already exist.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{CATALOG}`.`{SCHEMA}`")

# Protect an existing table from accidental replacement.
if spark.catalog.tableExists(TABLE_NAME):
    raise ValueError(
        "The Bronze sales table already exists. "
        "Use it for profiling, or review it before repeating ingestion."
    )

print("Checks passed. Ready to read the Sales worksheet.")


Checks passed. Ready to read the Sales worksheet.


In [0]:
# Open the workbook for reading without changing the file.
workbook = load_workbook(FILE_PATH, read_only=True, data_only=False) # this data_only parameter reads the formulas as well 

try:
    if SHEET_NAME not in workbook.sheetnames:
        raise ValueError(f"Worksheet not found: {SHEET_NAME}")

    worksheet = workbook[SHEET_NAME]
    excel_rows = worksheet.iter_rows(values_only=True) #this values_only return only value not cell object which could have formating or formula 
        # The first row contains the column names.
    headers = list(next(excel_rows))
    expected_headers = [
        "date", "order_id", "customer_id", "product_id",
        "units_sold", "total_amount", "store_id",
    ]
    if headers != expected_headers:
        raise ValueError(f"Unexpected Sales columns: {headers}")

    records = []
    for row_number, row in enumerate(excel_rows, start=2):
        # Preserve missing values and retain mixed data as text.
        values = [None if value is None else str(value) for value in row]
        records.append(tuple(values + [row_number]))

finally:
    # Always close the workbook after reading it.
    workbook.close()

source_count = len(records)
if source_count == 0:
    raise ValueError("The Sales worksheet has no data rows.")

print(f"Rows read from Excel: {source_count:,}")


#so far excel rows are saved as list of tuples on records object 

Rows read from Excel: 2,004


In [0]:
#first you need to understand that we use StructField to describe column , StructField("units_sold", StringType(), True) means column name is units_sold and its type is string and its nullable (True)
#StructType is a list of StructField
sales_schema = StructType(
    [StructField(column, StringType(), True) for column in headers]
    + [StructField("_source_row_number", LongType(), False)]
)
#now as we have schema let's create the datafram 
sales_df = spark.createDataFrame(records, schema=sales_schema)

sales_df.printSchema() #this is how we can see the schema of the dataframe
display(sales_df.orderBy("_source_row_number").limit(10))


root
 |-- date: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- units_sold: string (nullable = true)
 |-- total_amount: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- _source_row_number: long (nullable = false)



date,order_id,customer_id,product_id,units_sold,total_amount,store_id,_source_row_number
2021-08-23 00:00:00,ORD000001,CS_Sim_5,P10007,5,315,St_Pho_ A,2
2022-01-16 00:00:00,ORD000002,CS_Dav_37,P10073,4,136,St_Las_Ne,3
2021-07-19 00:00:00,ORD000003,CS_Dav_37,P10037,4,128,St_Las_Ne,4
2022-03-30 00:00:00,ORD000004,CS_Mat_12,P10082,2,60,St_Las_Ne,5
2022-01-29 00:00:00,ORD000005,CS_Bea_11,P10085,5,170,St_Atl_Ge,6
2020-12-22 00:00:00,ORD000006,CS_Kel_44,P10030,5,160,St_Hou_Te,7
2020-08-30 00:00:00,ORD000007,CS_Dav_15,P10037,6,192,St_Det_Mi,8
2021-01-24 00:00:00,ORD000008,CS_Don_33,P10031,5,170,St_Cha_No,9
2021-07-24 00:00:00,ORD000009,CS_Cal_22,P10066,3,96,St_Pho_ A,10
2022-06-04 00:00:00,ORD000010,CS_Com_17,P10032,6,198,St_Ind_In,11


In [0]:
# Create the table once; never silently overwrite or append.
(
    sales_df.write
    .format("delta")
    .saveAsTable(TABLE_NAME)
)

print(f"Created {TABLE_NAME}")


Created `sales_quality`.`bronze_layer`.`sales`


In [0]:
# Read the table back from Bronze.
bronze_sales = spark.table(TABLE_NAME)
saved_count = bronze_sales.count()

if saved_count != source_count:
    raise RuntimeError(
        f"Row count mismatch: {source_count} source rows, "
        f"{saved_count} saved rows. Review ingestion before continuing."
    )

# Show the reconciliation and a sample from the saved table.
ingestion_result = spark.createDataFrame(
    [(f"{CATALOG}.{SCHEMA}.sales", source_count, saved_count)],
    "table_name string, source_rows long, saved_rows long",
)

display(ingestion_result)
display(bronze_sales.orderBy("_source_row_number").limit(10))

table_name,source_rows,saved_rows
sales_quality.bronze_layer.sales,2004,2004


date,order_id,customer_id,product_id,units_sold,total_amount,store_id,_source_row_number
2021-08-23 00:00:00,ORD000001,CS_Sim_5,P10007,5,315,St_Pho_ A,2
2022-01-16 00:00:00,ORD000002,CS_Dav_37,P10073,4,136,St_Las_Ne,3
2021-07-19 00:00:00,ORD000003,CS_Dav_37,P10037,4,128,St_Las_Ne,4
2022-03-30 00:00:00,ORD000004,CS_Mat_12,P10082,2,60,St_Las_Ne,5
2022-01-29 00:00:00,ORD000005,CS_Bea_11,P10085,5,170,St_Atl_Ge,6
2020-12-22 00:00:00,ORD000006,CS_Kel_44,P10030,5,160,St_Hou_Te,7
2020-08-30 00:00:00,ORD000007,CS_Dav_15,P10037,6,192,St_Det_Mi,8
2021-01-24 00:00:00,ORD000008,CS_Don_33,P10031,5,170,St_Cha_No,9
2021-07-24 00:00:00,ORD000009,CS_Cal_22,P10066,3,96,St_Pho_ A,10
2022-06-04 00:00:00,ORD000010,CS_Com_17,P10032,6,198,St_Ind_In,11


%md
## Optional Ingestion Features

For this learning project, some production-oriented steps are optional:

- Source metadata such as file path, worksheet name and ingestion timestamp.
- Comparing source and saved row counts.
- Previewing the saved table after ingestion.

We can omit these steps to keep the notebook focused on the main ingestion process.

The important steps are:

1. Read the Excel worksheet.
2. Preserve missing and invalid values.
3. Create the Spark DataFrame with an explicit schema.
4. Save the DataFrame as a Bronze Delta table.
5. Keep `_source_row_number` for tracing validation issues back to Excel.

## Ingest the Remaining Tables

- Map the six worksheets to their destination Bronze tables.
- Check that the source files exist and destination tables do not already exist.
- Read each worksheet, preserve its raw values and add source metadata.
- Save the data as Delta tables in `sales_quality.bronze_layer`.
- Compare source and saved row counts to confirm ingestion.

In [0]:
# Each entry contains:
# (workbook filename, worksheet name, destination table name)
tables_to_ingest = [
    ("sales_practice.xlsx", "Customers", "customers"),
    ("sales_practice.xlsx", "Products", "products"),
    ("sales_practice.xlsx", "Stores", "stores"),
    ("sales_reference.xlsx", "product_categories", "product_categories"),
    ("sales_reference.xlsx", "store_locations", "store_locations"),
    ("sales_reference.xlsx", "category_aliases", "category_aliases"),
]

for filename, sheet_name, table_name in tables_to_ingest:
    file_path = f"{RAW_PATH}/{filename}"
    target = f"`{CATALOG}`.`{SCHEMA}`.`{table_name}`"

    # Confirm that the workbook is available.
    if not Path(file_path).is_file():
        raise FileNotFoundError(f"Missing workbook: {file_path}")

    # Stop if this table has already been ingested.
    if spark.catalog.tableExists(target):
        raise ValueError(
            f"Table {table_name} already exists. "
            "Review it before repeating ingestion."
        )

print("Checks passed. Ready to ingest the six remaining tables.")

Checks passed. Ready to ingest the six remaining tables.


In [0]:
# Collect the source and saved row counts.
results = []

for filename, sheet_name, table_name in tables_to_ingest:
    file_path = f"{RAW_PATH}/{filename}"
    target = f"`{CATALOG}`.`{SCHEMA}`.`{table_name}`"

    # Open the workbook without editing it.
    workbook = load_workbook(
        file_path,
        read_only=True,
        data_only=False,
    )

    try:
        # Confirm that the requested worksheet exists.
        if sheet_name not in workbook.sheetnames:
            raise ValueError(f"Missing worksheet: {sheet_name}")

        worksheet = workbook[sheet_name]
        excel_rows = worksheet.iter_rows(values_only=True)

        # Read column names from the first row.
        headers = list(next(excel_rows))

        # Column names must be populated and unique.
        if any(not isinstance(h, str) or not h.strip() for h in headers):
            raise ValueError(f"Invalid column names in {sheet_name}")

        if len(headers) != len(set(headers)):
            raise ValueError(f"Duplicate column names in {sheet_name}")

        # Read data rows and retain their Excel row numbers.
        records = []

        for row_number, row in enumerate(excel_rows, start=2):
            values = [
                None if value is None else str(value)
                for value in row
            ]

            records.append(tuple(values + [row_number]))

    finally:
        workbook.close()

    # Store business columns as text in Bronze.
    schema = StructType(
        [StructField(column, StringType(), True) for column in headers]
        + [StructField("_source_row_number", LongType(), False)]
    )

    df = spark.createDataFrame(records, schema=schema)


    # Save a new Delta table.
    df.write.format("delta").saveAsTable(target)

    # Confirm that the saved row count matches the source.
    source_count = len(records)
    saved_count = spark.table(target).count()

    if source_count != saved_count:
        raise RuntimeError(
            f"Row count mismatch for {table_name}: "
            f"{source_count} source rows, {saved_count} saved rows."
        )

    results.append((table_name, source_count, saved_count))
    print(f"Created {table_name}: {saved_count} rows")

Created customers: 49 rows
Created products: 100 rows
Created stores: 14 rows
Created product_categories: 6 rows
Created store_locations: 14 rows
Created category_aliases: 2 rows


In [0]:
display(
    spark.createDataFrame(
        results,
        "table_name string, source_rows long, saved_rows long",
    )
)

table_name,source_rows,saved_rows
customers,49,49
products,100,100
stores,14,14
product_categories,6,6
store_locations,14,14
category_aliases,2,2
